# BA840 Project Demo: Concussion Symptom Triage Chatbot (LLM-only vs Retrieval-only vs RAG)

This notebook implements **three “information channels”** for building a domain chatbot:

- **LLM-only:** the model answers from its general training (no documents)
- **Retrieval-only:** we **retrieve** relevant guideline passages and classify based on them only
- **RAG (Retrieval-Augmented Generation):** we retrieve passages and then ask an LLM to answer **with citations**

We’ll use a small set of **public CDC web pages** about concussion / mild traumatic brain injury (TBI).


## 0) Environment setup (Colab)

We will use:
- `requests` + `BeautifulSoup` to fetch and clean CDC web pages (HTML → text)
- `sentence-transformers` + `faiss` for **vector search** (retrieval)
- `transformers` for an open-source instruction-tuned LLM (generation)

> If you are running this in Google Colab:  
> **Runtime → Change runtime type → GPU** (recommended).

In [ ]:
# Install dependencies (Colab)
import sys, subprocess
IN_COLAB = "google.colab" in sys.modules

pkgs = [
    "pandas","numpy","matplotlib","tqdm",
    "requests","beautifulsoup4","lxml",
    "faiss-cpu","sentence-transformers",
    "transformers","accelerate","bitsandbytes"
]

if IN_COLAB:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])
    print("Installed packages.")
else:
    print("Not in Colab — assuming packages are already installed.")

Installed packages.


## 1) Imports + global settings

In this section we:
1. Import libraries
2. Create a local `data/` directory for cached documents and extracted text


In [ ]:
import os, re, time, textwrap
from dataclasses import dataclass
from typing import List, Dict, Tuple

import numpy as np
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt

from IPython.display import display

import requests
from bs4 import BeautifulSoup

import faiss
from sentence_transformers import SentenceTransformer

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

# Reproducibility
SEED = 7
np.random.seed(SEED)
torch.manual_seed(SEED)

DATA_DIR = "data"
TEXT_DIR = os.path.join(DATA_DIR, "text")
os.makedirs(TEXT_DIR, exist_ok=True)

print("Ready. Cache directory:", TEXT_DIR)


Ready. Cache directory: data/text


## 2) Domain labels + 3 prompt examples

We will predict one triage label:

- `SELF_CARE`
- `PRIMARY_CARE_SOON`
- `URGENT_CARE`
- `ER_NOW`
- `ABSTAIN` (out-of-domain or not enough support in the documents)

We will use **3 illustrative cases** throughout the notebook:

1. **Supported by evidence:** a CDC page explicitly contains the symptom and explicit emergency action language.
2. **Partly supported:** documents mention related symptoms and monitoring, but may not clearly map to one action.
3. **Not supported / out-of-domain:** symptoms that are unrelated with concussion and not mentioned in the documents.


In [ ]:
TRIAGE_LABELS = ["SELF_CARE", "PRIMARY_CARE_SOON", "URGENT_CARE", "ER_NOW", "ABSTAIN"]

CASES = [
    {
        "case_id": "A_supported",
        "label": "Supported by evidence",
        "user_symptoms": "I hit my head and now one pupil is larger than the other.",
        "why": "CDC danger-sign lists include 'one pupil larger than the other' with emergency instructions."
    },
    {
        "case_id": "B_partly",
        "label": "Partly supported",
        "user_symptoms": "I hit my head yesterday. Today I have a mild headache and feel a little dizzy, but I am alert.",
        "why": "CDC pages discuss symptoms and monitoring, but the exact triage level may not be explicit."
    },
    {
        "case_id": "C_not_supported",
        "label": "Not supported",
        "user_symptoms": "I fell ealier. I bit my tongue and my finger is purple now.",
        "why": "Our corpus is concussion/TBI guidance."
    },
]

pd.DataFrame(CASES)[["case_id","label","user_symptoms","why"]]

,case_id,label,user_symptoms,why
0,A_supported,Supported by evidence,I hit my head and now one pupil is larger than...,CDC danger-sign lists include 'one pupil large...
1,B_partly,Partly supported,I hit my head yesterday. Today I have a mild h...,"CDC pages discuss symptoms and monitoring, but..."
2,C_not_supported,Not supported,I fell ealier. I bit my tongue and my finger i...,Our corpus is concussion/TBI guidance.


## 3) LLM-only (no documents)

This is the simplest system:

> **Prompt → LLM → Answer**

But it is also often the *least trustworthy* in a domain setting, because the model can:
- hallucinate facts
- be overly confident
- follow prompt injection instructions
- give out-of-domain “medical” advice without evidence

### Output format
To keep outputs readable in class, we ask the model to output **exactly three lines**:

```
TRIAGE: <one of the labels>
RATIONALE: <1–2 sentences>
CITATIONS: NONE
```

### Model

Default: `Qwen/Qwen2.5-1.5B-Instruct` (good instruction-following; runs on Colab GPU).

If you hit memory issues, switch to:
- `Qwen/Qwen2.5-0.5B-Instruct` (smaller)

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
MAX_NEW_TOKENS = 220

assert torch.cuda.is_available(), "enable GPU: Runtime → Change runtime type → GPU"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.float16,
)

# Pipeline for convenience (we will still format prompts using the chat template)
gen = pipeline("text-generation", model=model, tokenizer=tokenizer, return_full_text=False)

# Avoid some common warnings
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id
model.generation_config.pad_token_id = tokenizer.pad_token_id

def make_chat_prompt(system: str, user: str) -> str:
    """Format (system,user) messages using the model's chat template.

    Why this matters:
    - Many "Instruct"/chat models expect special tokens and formatting.
    - If we pass a plain text prompt, format-following can become unreliable.
    """
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": user},
    ]
    # We will use the model's *chat template* (via tokenizer.apply_chat_template) so it behaves like a chat assistant.
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

print("Loaded:", MODEL_NAME)


Device set to use cuda:0


Loaded: Qwen/Qwen2.5-1.5B-Instruct


### 3.1 A tiny helper to call the LLM

We will define:
- `llm_generate(prompt, temperature)` — calls the model
- `llm_only_answer(user_symptoms, temperature)`

We also define a tiny parser **only for display**:
- extract the `TRIAGE:` line if present


In [ ]:
def llm_generate(system: str, user: str, temperature: float = 0.0, max_new_tokens: int = MAX_NEW_TOKENS) -> str:
    """Generate a response from the LLM using chat formatting."""
    prompt = make_chat_prompt(system, user)
    out = gen(
        prompt,
        max_new_tokens=max_new_tokens,
        do_sample=(temperature > 0.0),
        temperature=(temperature if temperature > 0.0 else 1.0),
    )[0]["generated_text"]
    return out.strip()

def parse_triage(text: str) -> str:
    """Extract a triage label from model output.

    We *expect* the model to output a line like:
      TRIAGE: ER_NOW

    In practice, models sometimes vary capitalization.
    So we parse case-insensitively and then normalize.
    """
    m = re.search(
        r"^\s*triage\s*:\s*(SELF_CARE|PRIMARY_CARE_SOON|URGENT_CARE|ER_NOW|ABSTAIN)\s*$",
        text,
        flags=re.I | re.M,
    )
    return m.group(1).upper() if m else "UNKNOWN"

def llm_only_answer(user_symptoms: str, temperature: float = 0.0) -> str:
    user = f"""USER_SYMPTOMS: {user_symptoms}

No documents are available in this mode.

Answer in the required 3-line format."""
    return llm_generate(SYSTEM_LLM_ONLY, user, temperature=temperature)

In [ ]:

SYSTEM_LLM_ONLY = """You are an educational triage assistant for concussion/head injury.

You must output EXACTLY 3 lines, no extra text:
TRIAGE: <ONE OF: SELF_CARE, PRIMARY_CARE_SOON, URGENT_CARE, ER_NOW, ABSTAIN>
RATIONALE: <1-2 sentences>
CITATIONS: NONE
"""

for c in CASES:
    print("\n" + "="*90)
    print(c["case_id"], "—", c["label"])
    print("USER_SYMPTOMS:", c["user_symptoms"])
    for t in (0.0, 0.7):
        ans = llm_only_answer(c["user_symptoms"], temperature=t)
        print(f"\n--- LLM-only (temp={t}) | parsed TRIAGE={parse_triage(ans)} ---")
        print(ans)



A_supported — Supported by evidence
USER_SYMPTOMS: I hit my head and now one pupil is larger than the other.

--- LLM-only (temp=0.0) | parsed TRIAGE=URGENT_CARE ---
Triage: URGENT_CARE

Rationale: The user has a significant change in pupil size following a head injury, which could indicate a more serious condition such as a brain bleed or intracranial hematoma that requires immediate medical attention.

Citations: None

--- LLM-only (temp=0.7) | parsed TRIAGE=URGENT_CARE ---
Triage: URGENT_CARE

Rationale: The user has experienced a significant change in their vision that could indicate a serious brain injury requiring immediate medical attention to prevent potential complications such as increased intracranial pressure or further damage.

Citations: None

B_partly — Partly supported
USER_SYMPTOMS: I hit my head yesterday. Today I have a mild headache and feel a little dizzy, but I am alert.

--- LLM-only (temp=0.0) | parsed TRIAGE=PRIMARY_CARE_SOON ---
Triage: PRIMARY_CARE_SOON

Rat

## 4) Build a small guideline corpus (CDC HTML pages)

We will use **CDC web pages**.

### What is a `doc_id`?
A `doc_id` is a short identifier we use in citations (e.g., `cdc_tbi_symptoms`).


In [ ]:
DOCS = [
    {
        "doc_id": "cdc_heads_up_concussion_basics",
        "title": "CDC HEADS UP — Concussion Basics",
        "url": "https://www.cdc.gov/heads-up/about/index.html",
    },
    {
        "doc_id": "cdc_heads_up_signs_symptoms",
        "title": "CDC HEADS UP — Signs and Symptoms of Concussion (includes danger signs)",
        "url": "https://www.cdc.gov/heads-up/signs-symptoms/index.html",
    },
    {
        "doc_id": "cdc_heads_up_responding",
        "title": "CDC HEADS UP — Responding to a Sports-related Concussion",
        "url": "https://www.cdc.gov/heads-up/response/index.html",
    },
    {
        "doc_id": "cdc_heads_up_recovery",
        "title": "CDC HEADS UP — What to do After a Concussion (recovery guidance)",
        "url": "https://www.cdc.gov/heads-up/guidelines/recovery-from-concussion.html",
    },
    {
        "doc_id": "cdc_tbi_about_mild",
        "title": "CDC TBI — About Mild TBI and Concussion",
        "url": "https://www.cdc.gov/traumatic-brain-injury/about/index.html",
    },
    {
        "doc_id": "cdc_tbi_symptoms",
        "title": "CDC TBI — Symptoms of Mild TBI and Concussion (includes danger signs)",
        "url": "https://www.cdc.gov/traumatic-brain-injury/signs-symptoms/index.html",
    },
    {
        "doc_id": "cdc_tbi_what_to_do",
        "title": "CDC TBI — What to Do After a Mild TBI or Concussion",
        "url": "https://www.cdc.gov/traumatic-brain-injury/response/index.html",
    },
]

pd.DataFrame(DOCS)[["doc_id","title","url"]]

,doc_id,title,url
0,cdc_heads_up_concussion_basics,CDC HEADS UP — Concussion Basics,https://www.cdc.gov/heads-up/about/index.html
1,cdc_heads_up_signs_symptoms,CDC HEADS UP — Signs and Symptoms of Concussio...,https://www.cdc.gov/heads-up/signs-symptoms/in...
2,cdc_heads_up_responding,CDC HEADS UP — Responding to a Sports-related ...,https://www.cdc.gov/heads-up/response/index.html
3,cdc_heads_up_recovery,CDC HEADS UP — What to do After a Concussion (...,https://www.cdc.gov/heads-up/guidelines/recove...
4,cdc_tbi_about_mild,CDC TBI — About Mild TBI and Concussion,https://www.cdc.gov/traumatic-brain-injury/abo...
5,cdc_tbi_symptoms,CDC TBI — Symptoms of Mild TBI and Concussion ...,https://www.cdc.gov/traumatic-brain-injury/sig...
6,cdc_tbi_what_to_do,CDC TBI — What to Do After a Mild TBI or Concu...,https://www.cdc.gov/traumatic-brain-injury/res...


### 4.1 HTML → text extraction (students should *see* this)

Web pages are full of menus, headers, and other “chrome”.
For retrieval, we want mostly the **main content**.

We will:
1. Download the raw HTML
2. Extract text from the `<main>` area when possible
3. Remove obvious non-content sections (`script`, `style`, `nav`, `header`, `footer`)


In [ ]:
def fetch_html(url: str, timeout: int = 30) -> str:
    r = requests.get(
        url,
        timeout=timeout,
        headers={"User-Agent": "Mozilla/5.0 (educational demo)"},
    )
    r.raise_for_status()
    return r.text

def html_to_text(html: str) -> str:
    """Convert (messy) HTML into cleaner text for our corpus.
    - Real webpages include menus, headers, footers, cookie banners, etc.
    - If we index that boilerplate, retrieval quality gets worse.
    """
    soup = BeautifulSoup(html, "lxml")

    # 1) Remove obvious non-content everywhere
    for tag in soup(["script", "style", "noscript", "svg", "iframe"]):
        tag.decompose()

    # 2) Pick a likely "main content" region
    main = (
        soup.find("main")
        or soup.find("article")
        or soup.find("div", {"id": "content"})
        or soup.body
        or soup
    )

    # 3) Remove common chrome inside that region
    for tag in main.find_all(["nav", "header", "footer", "aside", "form", "button"]):
        tag.decompose()

    # 4) Extract text with newlines (keeps headings and lists somewhat readable)
    text = main.get_text("\n", strip=True)

    # 5) Drop a few very common boilerplate lines (generic, not medical-specific)
    cleaned_lines = []
    for ln in text.splitlines():
        ln = ln.strip()
        if not ln:
            continue
        if re.search(r"skip\s+directly\s+to\s+content", ln, flags=re.I):
            continue
        if re.search(r"(facebook|twitter|linkedin|share|print)", ln, flags=re.I) and len(ln) < 40:
            continue
        cleaned_lines.append(ln)

    text = "\n".join(cleaned_lines)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

# Show one page: raw HTML vs extracted text
sample = DOCS[0]
print("URL:", sample["url"])

raw_html = fetch_html(sample["url"])
print("\n--- RAW HTML (first 500 chars) ---\n")
print(raw_html[:500])

clean = html_to_text(raw_html)
print("\n--- EXTRACTED TEXT (first 700 chars) ---\n")
print(clean[:700])


URL: https://www.cdc.gov/heads-up/about/index.html

--- RAW HTML (first 500 chars) ---

<!DOCTYPE html>
<html lang="en-us" class="cdc-theme-cdc cdc-page-type--content" >
<head>
	
<meta charset="UTF-8" />
<meta http-equiv="X-UA-Compatible" content="IE=Edge" />
<meta content="width=device-width, initial-scale=1, shrink-to-fit=no" name="viewport" />
<meta name="mobile-web-app-capable" content="yes" />
<meta name="apple-mobile-web-app-capable" content="yes" />

<link rel="apple-touch-icon" sizes="180x180" href="/TemplatePackage/5.0/img/favicon/apple-touch-icon.png">
<link re

--- EXTRACTED TEXT (first 700 chars) ---

January 7, 2025
Concussion Basics
Key points
A concussion is a type of traumatic brain injury.
The effects of a concussion can be serious.
A child with a concussion needs to be seen by a health care provider.
Concussions are brain injuries
A concussion is a type of traumatic brain injury—or TBI—caused by a bump, blow, or jolt to the head or by a hit to the body that causes the

### 4.2 Cache the corpus locally

For reproducibility, we store each extracted page as:

`data/text/<doc_id>.txt`

In [ ]:
def url_to_text(url: str) -> str:
    html = fetch_html(url)
    return html_to_text(html)

def build_corpus(docs: List[Dict[str,str]]) -> pd.DataFrame:
    rows = []
    for d in tqdm(docs):
        doc_id = d["doc_id"]
        path = os.path.join(TEXT_DIR, f"{doc_id}.txt")
        if not os.path.exists(path):
            txt = url_to_text(d["url"])
            with open(path, "w", encoding="utf-8") as f:
                f.write(txt)
        rows.append({
            "doc_id": doc_id,
            "title": d["title"],
            "source_url": d["url"],
            "file_path": path,
            "doc_type": "txt",
            "date_accessed": time.strftime("%Y-%m-%d"),
        })
    return pd.DataFrame(rows)

manifest = build_corpus(DOCS)
manifest

100%|██████████| 7/7 [00:00<00:00, 24164.71it/s]


,doc_id,title,source_url,file_path,doc_type,date_accessed
0,cdc_heads_up_concussion_basics,CDC HEADS UP — Concussion Basics,https://www.cdc.gov/heads-up/about/index.html,data/text/cdc_heads_up_concussion_basics.txt,txt,2026-01-07
1,cdc_heads_up_signs_symptoms,CDC HEADS UP — Signs and Symptoms of Concussio...,https://www.cdc.gov/heads-up/signs-symptoms/in...,data/text/cdc_heads_up_signs_symptoms.txt,txt,2026-01-07
2,cdc_heads_up_responding,CDC HEADS UP — Responding to a Sports-related ...,https://www.cdc.gov/heads-up/response/index.html,data/text/cdc_heads_up_responding.txt,txt,2026-01-07
3,cdc_heads_up_recovery,CDC HEADS UP — What to do After a Concussion (...,https://www.cdc.gov/heads-up/guidelines/recove...,data/text/cdc_heads_up_recovery.txt,txt,2026-01-07
4,cdc_tbi_about_mild,CDC TBI — About Mild TBI and Concussion,https://www.cdc.gov/traumatic-brain-injury/abo...,data/text/cdc_tbi_about_mild.txt,txt,2026-01-07
5,cdc_tbi_symptoms,CDC TBI — Symptoms of Mild TBI and Concussion ...,https://www.cdc.gov/traumatic-brain-injury/sig...,data/text/cdc_tbi_symptoms.txt,txt,2026-01-07
6,cdc_tbi_what_to_do,CDC TBI — What to Do After a Mild TBI or Concu...,https://www.cdc.gov/traumatic-brain-injury/res...,data/text/cdc_tbi_what_to_do.txt,txt,2026-01-07


### 4.3 Quick corpus preview

We’ll show:
- word count per document
- a short text snippet from each doc


In [ ]:
def read_doc(doc_id: str) -> str:
    path = os.path.join(TEXT_DIR, f"{doc_id}.txt")
    with open(path, "r", encoding="utf-8") as f:
        return f.read()

manifest["word_count"] = manifest["doc_id"].apply(lambda d: len(read_doc(d).split()))
display(manifest[["doc_id","word_count"]].sort_values("word_count", ascending=False))

# Snippet preview
for d in manifest["doc_id"].tolist():
    txt = read_doc(d)
    print("\n==", d, "==")
    print(txt[:350].replace("\n"," ") + ("..." if len(txt) > 350 else ""))

,doc_id,word_count
3,cdc_heads_up_recovery,2219
0,cdc_heads_up_concussion_basics,1061
2,cdc_heads_up_responding,947
1,cdc_heads_up_signs_symptoms,802
4,cdc_tbi_about_mild,731
6,cdc_tbi_what_to_do,731
5,cdc_tbi_symptoms,479



== cdc_heads_up_concussion_basics ==
January 7, 2025 Concussion Basics Key points A concussion is a type of traumatic brain injury. The effects of a concussion can be serious. A child with a concussion needs to be seen by a health care provider. Concussions are brain injuries A concussion is a type of traumatic brain injury—or TBI—caused by a bump, blow, or jolt to the head or by a hi...

== cdc_heads_up_signs_symptoms ==
Signs and Symptoms of Concussion Sept. 15, 2025 Key points Concussion signs and symptoms may not show up right away. Concussion signs can look different depending on the age of your child. Seek immediate emergency medical care if your child has any concussion danger signs. Signs and symptoms Concussion symptoms are what your child is feeling after ...

== cdc_heads_up_responding ==
Responding to a Sports-related Concussion Sept. 15, 2025 Key points Remove the athlete from sports participation right away if a concussion is suspected. Check the athlete for concussion d

## 5) Chunking: turning documents into retrievable passages

Retrieval systems usually search over **passages** (chunks), not entire documents.

Two parameters:
- `chunk_words`: how many words per chunk
- `overlap_words`: overlap between consecutive chunks

**These are very important parameters that could significantly affect your retrieval quality. Play around.**

Why overlap?
- prevents important sentences from being cut in half
- increases recall (but creates more chunks)


In [ ]:
def normalize_ws(s: str) -> str:
    s = s.replace("\u00a0", " ")
    s = re.sub(r"[ \t]+", " ", s)
    s = re.sub(r"\n{3,}", "\n\n", s)
    return s.strip()

def chunk_text_words(text: str, chunk_words: int = 220, overlap_words: int = 60) -> List[str]:
    text = normalize_ws(text)
    words = text.split()
    step = max(1, chunk_words - overlap_words)
    chunks = []
    for i in range(0, len(words), step):
        ch = " ".join(words[i:i+chunk_words]).strip()
        if len(ch) >= 200:
            chunks.append(ch)
    return chunks

# Show chunking on one doc with different parameters
doc_for_demo = "cdc_heads_up_signs_symptoms"
txt_demo = read_doc(doc_for_demo)

param_sets = [
    {"chunk_words": 140, "overlap_words": 30},
    {"chunk_words": 220, "overlap_words": 60},
    {"chunk_words": 320, "overlap_words": 80},
]

rows = []
for p in param_sets:
    chs = chunk_text_words(txt_demo, **p)
    rows.append({
        "chunk_words": p["chunk_words"],
        "overlap_words": p["overlap_words"],
        "num_chunks": len(chs),
        "first_chunk_preview": chs[0][:220] + ("..." if len(chs[0])>220 else ""),
    })

display(pd.DataFrame(rows))

for p in param_sets:
    chs = chunk_text_words(txt_demo, **p)
    print(f"\n=== chunk_words={p['chunk_words']} overlap_words={p['overlap_words']} | num_chunks={len(chs)} ===")
    for i in range(min(2, len(chs))):
        print(f"\n--- chunk {i} ---\n{chs[i][:700]}\n")

,chunk_words,overlap_words,num_chunks,first_chunk_preview
0,140,30,8,"Signs and Symptoms of Concussion Sept. 15, 202..."
1,220,60,5,"Signs and Symptoms of Concussion Sept. 15, 202..."
2,320,80,4,"Signs and Symptoms of Concussion Sept. 15, 202..."



=== chunk_words=140 overlap_words=30 | num_chunks=8 ===

--- chunk 0 ---
Signs and Symptoms of Concussion Sept. 15, 2025 Key points Concussion signs and symptoms may not show up right away. Concussion signs can look different depending on the age of your child. Seek immediate emergency medical care if your child has any concussion danger signs. Signs and symptoms Concussion symptoms are what your child is feeling after the injury. Concussion signs are what you see when your child is experiencing symptoms related to their injury. Experiencing concussion symptoms is part of the normal healing process. Concussion symptoms: 1 2 Are different for each person May change during recovery Are most severe right after the injury (usually 1 to 2 days after the injury) Differ


--- chunk 1 ---
after the injury) Differ depending on the age of your child Keep in mind that concussion signs and symptoms may not show up right away—they may take hours or days to appear or be noticed. Spotting a possible

### Chunk embeddings

We will:
1. Create chunks for every document
2. Embed each chunk using a sentence embedding model (`all-MiniLM-L6-v2`)
3. Build a FAISS index for fast similarity search

In [ ]:
@dataclass
class DocChunk:
    doc_id: str
    chunk_id: str
    text: str

# Build chunks for the full corpus (use the *default* chunking parameters)
CHUNK_WORDS = 220
OVERLAP_WORDS = 60

chunks: List[DocChunk] = []
for doc_id in manifest["doc_id"].tolist():
    txt = read_doc(doc_id)
    chs = chunk_text_words(txt, chunk_words=CHUNK_WORDS, overlap_words=OVERLAP_WORDS)
    for j, ch in enumerate(chs):
        chunks.append(DocChunk(doc_id=doc_id, chunk_id=f"{doc_id}__{j}", text=ch))

print("Num documents:", len(manifest))
print("Num chunks:", len(chunks))

# Embed chunks + FAISS
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
embedder = SentenceTransformer(EMBED_MODEL)

texts = [c.text for c in chunks]
X = np.array(embedder.encode(texts, normalize_embeddings=True, show_progress_bar=True), dtype=np.float32)

index = faiss.IndexFlatIP(X.shape[1])  # cosine similarity (normalized)
index.add(X)

Num documents: 7
Num chunks: 45


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

## 6) Build a retriever (vector search)

Implement `retrieve(query, k)`

### What is `k`?
When we retrieve, we ask for the **top-k most similar chunks**:
- `k=2` → less context / less noise, but may miss key info
- `k=5` → more context / more recall, but may include irrelevant chunks


In [ ]:
def retrieve(query: str, k: int = 5, per_doc_cap: int = 2) -> List[Tuple[DocChunk, float]]:
    q = np.array(embedder.encode([query], normalize_embeddings=True), dtype=np.float32)
    scores, idx = index.search(q, 25)  # retrieve more than k, then diversify
    scores = scores[0].tolist()
    idx = idx[0].tolist()

    out = []
    counts = {}
    for i, sc in zip(idx, scores):
        ch = chunks[i]
        counts[ch.doc_id] = counts.get(ch.doc_id, 0) + 1
        if counts[ch.doc_id] <= per_doc_cap:
            out.append((ch, float(sc)))
        if len(out) >= k:
            break
    return out

def show_hits(hits: List[Tuple[DocChunk, float]]) -> pd.DataFrame:
    rows = []
    for rank, (ch, sc) in enumerate(hits, start=1):
        title = manifest.loc[manifest["doc_id"] == ch.doc_id, "title"].iloc[0]
        rows.append({
            "rank": rank,
            "doc_id": ch.doc_id,
            "title": title,
            "score": round(sc, 3),
            "chunk_id": ch.chunk_id,
            "snippet": ch.text[:220].replace("\n"," ") + ("..." if len(ch.text)>220 else ""),
        })
    return pd.DataFrame(rows)

### 6.1 What does retrieval do? (k=2 vs k=5)

For each case, we will retrieve evidence and inspect the passages.

- Did retrieval find the right doc?
- Did it find the right *passage*?
- How does changing `k` change the evidence set?


In [ ]:
for c in CASES:
    print("\n" + "="*100)
    print(c["case_id"], "—", c["label"])
    print("USER_SYMPTOMS:", c["user_symptoms"])
    for k in (2, 5):
        hits = retrieve(c["user_symptoms"], k=k)
        print(f"\nTop-{k} hits:")
        display(show_hits(hits))


A_supported — Supported by evidence
USER_SYMPTOMS: I hit my head and now one pupil is larger than the other.

Top-2 hits:


,rank,doc_id,title,score,chunk_id,snippet
0,1,cdc_tbi_symptoms,CDC TBI — Symptoms of Mild TBI and Concussion ...,0.427,cdc_tbi_symptoms__2,blood clot that crowds the brain against the s...
1,2,cdc_heads_up_signs_symptoms,CDC HEADS UP — Signs and Symptoms of Concussio...,0.323,cdc_heads_up_signs_symptoms__2,4 ) A Irritability and crying more than usual ...



Top-5 hits:


,rank,doc_id,title,score,chunk_id,snippet
0,1,cdc_tbi_symptoms,CDC TBI — Symptoms of Mild TBI and Concussion ...,0.427,cdc_tbi_symptoms__2,blood clot that crowds the brain against the s...
1,2,cdc_heads_up_signs_symptoms,CDC HEADS UP — Signs and Symptoms of Concussio...,0.323,cdc_heads_up_signs_symptoms__2,4 ) A Irritability and crying more than usual ...
2,3,cdc_tbi_about_mild,CDC TBI — About Mild TBI and Concussion,0.291,cdc_tbi_about_mild__3,TBI or Concussion for tips to help aid your re...
3,4,cdc_heads_up_signs_symptoms,CDC HEADS UP — Signs and Symptoms of Concussio...,0.279,cdc_heads_up_signs_symptoms__3,"slowed reaction times Mood, behavior, or perso..."
4,5,cdc_tbi_symptoms,CDC TBI — Symptoms of Mild TBI and Concussion ...,0.265,cdc_tbi_symptoms__1,light or noise Dizziness or balance problems F...



B_partly — Partly supported
USER_SYMPTOMS: I hit my head yesterday. Today I have a mild headache and feel a little dizzy, but I am alert.

Top-2 hits:


,rank,doc_id,title,score,chunk_id,snippet
0,1,cdc_tbi_symptoms,CDC TBI — Symptoms of Mild TBI and Concussion ...,0.556,cdc_tbi_symptoms__1,light or noise Dizziness or balance problems F...
1,2,cdc_tbi_symptoms,CDC TBI — Symptoms of Mild TBI and Concussion ...,0.540,cdc_tbi_symptoms__0,"Symptoms of Mild TBI and Concussion Sept. 15, ..."


... (2 outputs left out)


C_not_supported — Not supported
USER_SYMPTOMS: I fell ealier. I bit my tongue and my finger is purple now.

Top-2 hits:


,rank,doc_id,title,score,chunk_id,snippet
0,1,cdc_tbi_symptoms,CDC TBI — Symptoms of Mild TBI and Concussion ...,0.218,cdc_tbi_symptoms__2,blood clot that crowds the brain against the s...
1,2,cdc_tbi_about_mild,CDC TBI — About Mild TBI and Concussion,0.165,cdc_tbi_about_mild__2,provider. Contact your healthcare provider as ...



Top-5 hits:


,rank,doc_id,title,score,chunk_id,snippet
0,1,cdc_tbi_symptoms,CDC TBI — Symptoms of Mild TBI and Concussion ...,0.218,cdc_tbi_symptoms__2,blood clot that crowds the brain against the s...
1,2,cdc_tbi_about_mild,CDC TBI — About Mild TBI and Concussion,0.165,cdc_tbi_about_mild__2,provider. Contact your healthcare provider as ...
2,3,cdc_tbi_what_to_do,CDC TBI — What to Do After a Mild TBI or Concu...,0.164,cdc_tbi_what_to_do__0,What to Do After a Mild TBI or Concussion Sept...
3,4,cdc_tbi_what_to_do,CDC TBI — What to Do After a Mild TBI or Concu...,0.161,cdc_tbi_what_to_do__2,even if you still have some mild symptoms. If ...
4,5,cdc_heads_up_signs_symptoms,CDC HEADS UP — Signs and Symptoms of Concussio...,0.159,cdc_heads_up_signs_symptoms__2,4 ) A Irritability and crying more than usual ...


## 7) Retrieval-only

In this mode, the system **does not answer**.  
It only returns **retrieved passages + doc IDs**.

In [ ]:
def retrieval_only(user_symptoms: str, k: int = 5) -> List[Tuple[DocChunk, float]]:
    return retrieve(user_symptoms, k=k)

def print_evidence(hits: List[Tuple[DocChunk, float]], max_chars: int = 600):
    for rank, (ch, sc) in enumerate(hits, start=1):
        print(f"\n--- Evidence #{rank} | doc_id={ch.doc_id} | score={sc:.3f} ---")
        print(ch.text[:max_chars])
        if len(ch.text) > max_chars:
            print("...[truncated]")

# Demo Mode 0 on the 3 cases
for c in CASES:
    print("\n" + "="*100)
    print("MODE 0 (Retrieval-only) —", c["case_id"], c["label"])
    hits = retrieval_only(c["user_symptoms"], k=5)
    display(show_hits(hits))
    print("\nEvidence passages (verbatim):")
    print_evidence(hits, max_chars=500)


MODE 0 (Retrieval-only) — A_supported Supported by evidence


,rank,doc_id,title,score,chunk_id,snippet
0,1,cdc_tbi_symptoms,CDC TBI — Symptoms of Mild TBI and Concussion ...,0.427,cdc_tbi_symptoms__2,blood clot that crowds the brain against the s...
1,2,cdc_heads_up_signs_symptoms,CDC HEADS UP — Signs and Symptoms of Concussio...,0.323,cdc_heads_up_signs_symptoms__2,4 ) A Irritability and crying more than usual ...
2,3,cdc_tbi_about_mild,CDC TBI — About Mild TBI and Concussion,0.291,cdc_tbi_about_mild__3,TBI or Concussion for tips to help aid your re...
3,4,cdc_heads_up_signs_symptoms,CDC HEADS UP — Signs and Symptoms of Concussio...,0.279,cdc_heads_up_signs_symptoms__3,"slowed reaction times Mood, behavior, or perso..."
4,5,cdc_tbi_symptoms,CDC TBI — Symptoms of Mild TBI and Concussion ...,0.265,cdc_tbi_symptoms__1,light or noise Dizziness or balance problems F...



Evidence passages (verbatim):

--- Evidence #1 | doc_id=cdc_tbi_symptoms | score=0.427 ---
blood clot that crowds the brain against the skull can develop. The people checking on you should call 9-1-1 or take you to an emergency department right away if you: Have a headache that gets worse and does not go away Experience weakness, numbness, decreased coordination, convulsions, or seizures Vomit repeatedly Have slurred speech or unusual behavior Have one pupil (the black part in the middle of the eye) larger than the other Cannot recognize people or places, get confused, restless, or ag
...[truncated]

--- Evidence #2 | doc_id=cdc_heads_up_signs_symptoms | score=0.323 ---
4 ) A Irritability and crying more than usual Seeking more comfort than usual, like needing to be held Refusing to nurse or eat Changes in sleeping patterns, like sleeping more or less than usual Not wanting to engage in play or usual activities Needing more help than usual Showing behavior changes, such as more temper

,rank,doc_id,title,score,chunk_id,snippet
0,1,cdc_tbi_symptoms,CDC TBI — Symptoms of Mild TBI and Concussion ...,0.556,cdc_tbi_symptoms__1,light or noise Dizziness or balance problems F...
1,2,cdc_tbi_symptoms,CDC TBI — Symptoms of Mild TBI and Concussion ...,0.540,cdc_tbi_symptoms__0,"Symptoms of Mild TBI and Concussion Sept. 15, ..."
2,3,cdc_heads_up_signs_symptoms,CDC HEADS UP — Signs and Symptoms of Concussio...,0.537,cdc_heads_up_signs_symptoms__2,4 ) A Irritability and crying more than usual ...
3,4,cdc_tbi_about_mild,CDC TBI — About Mild TBI and Concussion,0.510,cdc_tbi_about_mild__0,"About Mild TBI and Concussion Sept. 15, 2025 K..."
4,5,cdc_tbi_about_mild,CDC TBI — About Mild TBI and Concussion,0.470,cdc_tbi_about_mild__1,and test you to see if you have any problems w...



Evidence passages (verbatim):

--- Evidence #1 | doc_id=cdc_tbi_symptoms | score=0.556 ---
light or noise Dizziness or balance problems Feeling tired, no energy Headaches Nausea or vomiting (early on) Vision problems Thinking and Remembering Attention or concentration problems Feeling slowed down Foggy or groggy Problems with short- or long-term memory Trouble thinking clearly Social or Emotional Anxiety or nervousness Irritability or easily angered Feeling more emotional Sadness Sleep Sleeping less than usual Sleeping more than usual Trouble falling asleep Complications and when to s
...[truncated]

--- Evidence #2 | doc_id=cdc_tbi_symptoms | score=0.540 ---
Symptoms of Mild TBI and Concussion Sept. 15, 2025 Key Points Symptoms of mild TBI and concussion are different for each person. Symptoms of mild TBI and concussion may affect how you feel, think, act or sleep. Seek immediate emergency medical care if you have any danger signs. Symptoms Some mild TBI and concussion symptoms may a

,rank,doc_id,title,score,chunk_id,snippet
0,1,cdc_tbi_symptoms,CDC TBI — Symptoms of Mild TBI and Concussion ...,0.218,cdc_tbi_symptoms__2,blood clot that crowds the brain against the s...
1,2,cdc_tbi_about_mild,CDC TBI — About Mild TBI and Concussion,0.165,cdc_tbi_about_mild__2,provider. Contact your healthcare provider as ...
2,3,cdc_tbi_what_to_do,CDC TBI — What to Do After a Mild TBI or Concu...,0.164,cdc_tbi_what_to_do__0,What to Do After a Mild TBI or Concussion Sept...
3,4,cdc_tbi_what_to_do,CDC TBI — What to Do After a Mild TBI or Concu...,0.161,cdc_tbi_what_to_do__2,even if you still have some mild symptoms. If ...
4,5,cdc_heads_up_signs_symptoms,CDC HEADS UP — Signs and Symptoms of Concussio...,0.159,cdc_heads_up_signs_symptoms__2,4 ) A Irritability and crying more than usual ...



Evidence passages (verbatim):

--- Evidence #1 | doc_id=cdc_tbi_symptoms | score=0.218 ---
blood clot that crowds the brain against the skull can develop. The people checking on you should call 9-1-1 or take you to an emergency department right away if you: Have a headache that gets worse and does not go away Experience weakness, numbness, decreased coordination, convulsions, or seizures Vomit repeatedly Have slurred speech or unusual behavior Have one pupil (the black part in the middle of the eye) larger than the other Cannot recognize people or places, get confused, restless, or ag
...[truncated]

--- Evidence #2 | doc_id=cdc_tbi_about_mild | score=0.165 ---
provider. Contact your healthcare provider as soon as you can if you think you have a mild TBI or concussion. Getting care after a mild TBI or concussion can help speed your recovery. You may consider using telemedicine or telehealth services if you are unable to visit a healthcare provider in person. Telemedicine and telehealt

## 8) RAG

RAG is:

> **Retrieve → Generate**

We will:
1. Retrieve top-k passages
2. Put them into an `EVIDENCE` block
3. Ask the LLM to output (exactly 3 lines):

```
TRIAGE: <label>
RATIONALE: <1–2 sentences>
CITATIONS: <comma-separated doc_ids>
```


In [ ]:
def build_evidence_block(hits: List[Tuple[DocChunk, float]], max_chars_per_chunk: int = 900) -> str:
    """Create an evidence block shown to the LLM.

    We include doc_id and title so the model can cite correctly.
    """
    parts = []
    for ch, sc in hits:
        title = manifest.loc[manifest["doc_id"] == ch.doc_id, "title"].iloc[0]
        parts.append(f"[DOC {ch.doc_id} | {title}] {ch.text[:max_chars_per_chunk]}")
    return "\n\n".join(parts)

SYSTEM_RAG = """You are an educational triage assistant for concussion/head injury.

You MUST base your answer ONLY on:
- USER_SYMPTOMS
- EVIDENCE (retrieved passages)

Hard rules:
- Do NOT use outside medical knowledge.
- Do NOT add symptoms that are not explicitly in USER_SYMPTOMS.
- If USER_SYMPTOMS are not about a head injury / concussion, output ABSTAIN (out-of-domain).
- If the EVIDENCE does not clearly recommend an action for the user's described symptoms, output ABSTAIN.

Label meanings (map EVIDENCE → label):
- ER_NOW: evidence says call 911 / go to ER / emergency department.
- URGENT_CARE: evidence says seek care immediately / right away, but not explicitly ER/911.
- PRIMARY_CARE_SOON: evidence says contact/see a healthcare provider/doctor soon.
- SELF_CARE: evidence supports home care / rest / monitor, with no urgent escalation.
- ABSTAIN: not supported or missing.

Output format (MUST follow exactly):
TRIAGE: <SELF_CARE | PRIMARY_CARE_SOON | URGENT_CARE | ER_NOW | ABSTAIN>
RATIONALE: <1-2 sentences grounded in EVIDENCE>
CITATIONS: <comma-separated doc_ids OR NONE>
"""

def rag_answer(user_symptoms: str, k: int = 5, temperature: float = 0.0) -> str:
    hits = retrieve(user_symptoms, k=k)
    valid_doc_ids = sorted(list({h[0].doc_id for h in hits}))
    evidence = build_evidence_block(hits)

    user = f"""USER_SYMPTOMS: {user_symptoms}

EVIDENCE:
{evidence}

CITATIONS must be chosen ONLY from these doc_ids (or NONE if ABSTAIN):
{', '.join(valid_doc_ids)}
"""
    return llm_generate(SYSTEM_RAG, user, temperature=temperature)

# Demo RAG across k and temperature for the 3 cases
rows = []
for c in CASES:
    for k in (2, 5):
        for t in (0.0, 0.7):
            out = rag_answer(c["user_symptoms"], k=k, temperature=t)
            rows.append({
                "case_id": c["case_id"],
                "k": k,
                "temp": t,
                "triage_parsed": parse_triage(out),
                "output_preview": out[:180].replace("\n"," | ") + ("..." if len(out)>180 else "")
            })
df = pd.DataFrame(rows)
display(df)


,case_id,k,temp,triage_parsed,output_preview
0,A_supported,2,0.0,ER_NOW,TRIAGE: ER_NOW | RATIONALE: The symptom of one...
1,A_supported,2,0.7,ER_NOW,TRIAGE: ER_NOW | RATIONALE: Based on the provi...
2,A_supported,5,0.0,ABSTAIN,TRIAGE: ABSTAIN | RATIONALE: The provided symp...
3,A_supported,5,0.7,ABSTAIN,TRIAGE: ABSTAIN | RATIONALE: The provided info...
4,B_partly,2,0.0,SELF_CARE,TRIAGE: SELF_CARE | RATIONALE: The symptoms de...
5,B_partly,2,0.7,SELF_CARE,TRIAGE: SELF_CARE | RATIONALE: The symptoms de...
6,B_partly,5,0.0,SELF_CARE,TRIAGE: SELF_CARE | RATIONALE: The symptoms de...
7,B_partly,5,0.7,SELF_CARE,TRIAGE: SELF_CARE | RATIONALE: Based on the sy...
8,C_not_supported,2,0.0,ABSTAIN,TRIAGE: ABSTAIN | RATIONALE: The provided symp...
9,C_not_supported,2,0.7,URGENT_CARE,TRIAGE: URGENT_CARE | RATIONALE: The user ment...
